In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

$\text{\textcolor{green}{Load}}$

In [ ]:
df_load = pd.read_csv(
    ROOT / "data/archive/archive/building_consumption.csv",
    parse_dates=["timestamp"]
)

filtered = df_load[
    (df_load["campus_id"] == 1) &
    (df_load["timestamp"] >= "2021-05-14 07:30:00") &
    (df_load["timestamp"] <= "2022-04-10 23:30:00")
]

total_load = (
    filtered
    .groupby("timestamp")["consumption"]
    .sum()
    .reset_index(name="total_consumption")
)

total_load["timestamp"] = pd.to_datetime(total_load["timestamp"])
total_load = total_load.set_index("timestamp")

total_load_30min = total_load.resample("30min").mean()

load_scale = 0.1
total_load_30min["total_consumption"] *= load_scale

total_load_30min["date"] = total_load_30min.index.date
total_load_30min["month_day"] = total_load_30min.index.strftime("%m-%d")

load_counts = total_load_30min.groupby("date")["total_consumption"].count()
load_complete_days = load_counts[load_counts == 48].index

load_days_df = total_load_30min[
    total_load_30min["date"].isin(load_complete_days)
].copy()

load_day_profiles = []

for date, day_df in load_days_df.groupby("date"):
    day_df = day_df.sort_index()

    load_day_profiles.append({
        "load_date": date,
        "month_day": pd.to_datetime(date).strftime("%m-%d"),
        "Load_profile": day_df["total_consumption"].to_numpy()
    })

load_profiles_df = pd.DataFrame(load_day_profiles)

print(load_profiles_df)

$\text{\textcolor{green}{Generation}}$

In [ ]:
DATA_DIR = ROOT / "data/complete_dataframe2_30min.csv"

df_gen = pd.read_csv(
    DATA_DIR,
    delimiter=",",
    decimal=".",
    parse_dates=["ts"],
    index_col="ts"
)

df_agg_gen = df_gen.copy()

df_agg_gen["PV_total"] = df_agg_gen[
    ["B117_m", "B319_m", "B330_1_m", "B330_2_m", "B330_3_m", "B716_m", "B715_m"]
].sum(axis=1, min_count=1)

df_agg_gen["Wind_total"] = df_agg_gen[
    ["Aircon_WT Power_m", "Gaia_WT Power_m"]
].sum(axis=1, min_count=1)

df_agg_gen.dropna(subset=["PV_total", "Wind_total"], how="any", inplace=True)

df_agg_gen["date"] = df_agg_gen.index.date
df_agg_gen["month_day"] = df_agg_gen.index.strftime("%m-%d")

counts_per_day = df_agg_gen.groupby("date").size()
complete_days = counts_per_day[counts_per_day == 48].index

df_agg_gen = df_agg_gen[df_agg_gen["date"].isin(complete_days)].copy()

gen_day_profiles = []

for date, day_df in df_agg_gen.groupby("date"):
    day_df = day_df.sort_index()

    pv_profile = day_df["PV_total"].to_numpy()
    wind_profile = day_df["Wind_total"].to_numpy()
    renewable_profile = pv_profile + wind_profile

    gen_day_profiles.append({
        "gen_date": date,
        "month_day": pd.to_datetime(date).strftime("%m-%d"),
        "PV_profile": pv_profile,
        "Wind_profile": wind_profile,
        "Renewable_profile": renewable_profile
    })

gen_profiles_df = pd.DataFrame(gen_day_profiles)
#print(gen_day_profiles)

$\text{\textcolor{green}{Day-Ahead Prices}}$

In [ ]:
df_price = pd.read_csv(ROOT / "data/DayAheadData_fulltimeperiod.csv")

df_price = df_price[(df_price["ts"] >= "2025-05-14 07:30:00") &
    (df_price["ts"] <= "2026-04-10 23:30:00")]

df_price["ts"] = pd.to_datetime(df_price["ts"])
df_price = df_price.set_index("ts")

df_price["date"] = df_price.index.date
df_price["month_day"] = df_price.index.strftime("%m-%d")

counts_per_day_price = df_price.groupby("date").size()
complete_days_price = counts_per_day_price[counts_per_day_price == 48].index

df_price = df_price[df_price["date"].isin(complete_days_price)].copy()

price_day_profiles = []

for date, day_df in df_price.groupby("date"):
    day_df = day_df.sort_index()
    
    price_day_profiles.append({
        "gen_date": date,
        "month_day": pd.to_datetime(date).strftime("%m-%d"),
        "Price_profile": day_df["DayAheadPriceDKK"].to_numpy()
    })

price_profiles_df = pd.DataFrame(price_day_profiles)


In [ ]:
paired_day_profiles = load_profiles_df.merge(gen_profiles_df, on="month_day", how="inner").merge(price_profiles_df, on="month_day", how="inner")

paired_day_profiles = (paired_day_profiles.sort_values("month_day").reset_index(drop=True))

paired_day_profiles.columns


In [ ]:
# Visualize load
time_labels = pd.date_range("00:00", periods=48, freq="30min").strftime("%H:%M")
    # Power of load
plt.figure(figsize=(10, 6))

for profile in paired_day_profiles["Load_profile"]:
    plt.plot(time_labels, profile, alpha=0.25)

plt.xticks(time_labels[::4], rotation=45)
plt.xlabel("Time of day")
plt.ylabel("Load [kW]")
plt.title("Daily load profiles")
plt.grid(True)
plt.show()

    # Energy of load across all paired days 

total_energy_kWh = sum(
sum(profile) * 0.5
    for profile in paired_day_profiles["Load_profile"])
print(f"Total energy demand: {total_energy_kWh:.0f} kWh")

n_days = len(paired_day_profiles)
avg_daily_energy = total_energy_kWh / n_days
print(f"Average daily energy: {avg_daily_energy:.1f} kWh/day")

avg_power_kW = avg_daily_energy / 24
print(f"Equivalent constant load: {avg_power_kW:.1f} kW")

# Visualize generation 

plt.figure(figsize=(10, 6))

for profile in paired_day_profiles["Renewable_profile"]:
    plt.plot(time_labels, profile, alpha=0.25)

plt.xticks(time_labels[::4], rotation=45)
plt.xlabel("Time of day")
plt.ylabel("Generation [kW]")
plt.title("Daily generation profiles")
plt.grid(True)
plt.show()


In [ ]:
def solve_no_battery_day_from_pair(row, dT=0.5):
    load = np.asarray(row["Load_profile"])
    pv = np.asarray(row["PV_profile"])
    wind = np.asarray(row["Wind_profile"])
    price = np.asarray(row["Price_profile"])

    generation = pv + wind

    if not (len(load) == len(generation) == len(price) == 48):
        return {"status": "bad_input: length mismatch"}

    if not np.isfinite(load).all():
        return {"status": "bad_input: non-finite load"}

    if not np.isfinite(generation).all():
        return {"status": "bad_input: non-finite generation"}

    if not np.isfinite(price).all():
        return {"status": "bad_input: non-finite price"}

    direct_renewable_use = np.minimum(load, generation)
    grid_import = np.maximum(load - generation, 0)
    curtailment = np.maximum(generation - load, 0)

    total_load_kWh = load.sum() * dT
    total_generation_kWh = generation.sum() * dT
    self_consumed_renewables_kWh = direct_renewable_use.sum() * dT
    grid_import_kWh = grid_import.sum() * dT
    curtailment_kWh = curtailment.sum() * dT

    grid_cost = np.sum(grid_import * price) * dT

    return {
        "status": "optimal",
        "load_date": row["load_date"],
        "gen_date": row["gen_date_x"],
        "month_day": row["month_day"],

        "total_load_kWh": total_load_kWh,
        "total_generation_kWh": total_generation_kWh,
        "self_consumed_renewables_kWh": self_consumed_renewables_kWh,
        "grid_import_kWh": grid_import_kWh,
        "curtailment_kWh": curtailment_kWh,
        "grid_cost": grid_cost,

        "self_sufficiency_pct": (
            self_consumed_renewables_kWh / total_load_kWh
        ) * 100,

        "self_consumption_pct": (
            self_consumed_renewables_kWh / total_generation_kWh
        ) * 100 if total_generation_kWh > 0 else np.nan,

        "Load_profile": load,
        "Generation_profile": generation,
        "Direct_renewable_use_profile": direct_renewable_use,
        "Grid_import_profile": grid_import,
        "Curtailment_profile": curtailment,
        "Price_profile": price,
    }

In [ ]:
baseline_results = []

for _, row in paired_day_profiles.iterrows():
    out = solve_no_battery_day_from_pair(row, dT=0.5)
    baseline_results.append(out)

baseline_results_df = pd.DataFrame(baseline_results)

In [ ]:
# Results correspond to energies and costs spanned across a year

baseline_summary = pd.Series({
    "total_days": len(baseline_results_df),
    "total_load_kWh": baseline_results_df["total_load_kWh"].sum(),
    "total_generation_kWh": baseline_results_df["total_generation_kWh"].sum(),
    "self_consumed_renewables_kWh": baseline_results_df["self_consumed_renewables_kWh"].sum(),
    "grid_import_kWh": baseline_results_df["grid_import_kWh"].sum(),
    "curtailment_kWh": baseline_results_df["curtailment_kWh"].sum(),
    "grid_cost_DKK": baseline_results_df["grid_cost"].sum(),
})

baseline_summary["self_sufficiency_pct"] = (
    baseline_summary["self_consumed_renewables_kWh"]
    / baseline_summary["total_load_kWh"]
) * 100

baseline_summary["self_consumption_pct"] = (
    baseline_summary["self_consumed_renewables_kWh"]
    / baseline_summary["total_generation_kWh"]
) * 100

baseline_summary.round(2)

In [ ]:
daily_baseline_summary = baseline_results_df[
    [
        "month_day",
        "load_date",
        "gen_date",
        "total_load_kWh",
        "total_generation_kWh",
        "self_consumed_renewables_kWh",
        "grid_import_kWh",
        "curtailment_kWh",
        "grid_cost",
        "self_sufficiency_pct",
        "self_consumption_pct",
    ]
].round(2)

daily_baseline_summary